In [1]:
import torch

from floodvision.losses.masked_bce import MaskedBCELoss
from floodvision.losses.dice import MaskedDiceLoss
from floodvision.losses.combined import BCEDiceLoss

from floodvision.metrics.segmentation import (
    SegmentationMetrics
)

In [3]:
logits = torch.tensor(
    [[[[3.0, -3.0, 2.0],
       [-2.0, 1.0, -1.0],
       [4.0, -4.0, 2.0]]]],
    requires_grad=True
)

targets = torch.tensor(
    [[[[1.0, 0.0, 1.0],
       [0.0, 0.0, 1.0],
       [-1.0, -1.0, 1.0]]]]
)

print("Logits shape:", logits.shape)
print("Targets shape:", targets.shape)

Logits shape: torch.Size([1, 1, 3, 3])
Targets shape: torch.Size([1, 1, 3, 3])


In [4]:
bce = MaskedBCELoss()

weighted_bce = MaskedBCELoss(
    pos_weight=2.0
)

dice = MaskedDiceLoss()

combined = BCEDiceLoss()

print("BCE:", bce(logits, targets).item())

print(
    "Weighted BCE:",
    weighted_bce(logits, targets).item()
)

print("Dice Loss:", dice(logits, targets).item())

print(
    "Combined Loss:",
    combined(logits, targets).item()
)

BCE: 0.4434974789619446
Weighted BCE: 0.6743124723434448
Dice Loss: 0.242942214012146
Combined Loss: 0.3432198464870453


In [5]:
logits.grad = None

loss = combined(logits, targets)

loss.backward()

print("Loss:", loss.item())

print("Gradients:")
print(logits.grad)

assert torch.isfinite(logits.grad).all()

assert (
    logits.grad[targets == -1] == 0
).all()

print("Gradient validation successful!")

Loss: 0.3432198464870453
Gradients:
tensor([[[[-0.0070,  0.0056, -0.0168],
          [ 0.0136,  0.0617, -0.0677],
          [ 0.0000,  0.0000, -0.0168]]]])
Gradient validation successful!


In [6]:
metrics = SegmentationMetrics(
    threshold=0.5
)

metrics.update(logits, targets)

print("TP:", metrics.tp)
print("TN:", metrics.tn)
print("FP:", metrics.fp)
print("FN:", metrics.fn)

for name, value in metrics.compute().items():

    print(f"{name}: {value:.4f}")

TP: 3
TN: 2
FP: 1
FN: 1
accuracy: 0.7143
precision: 0.7500
recall: 0.7500
iou: 0.6000
dice: 0.7500
specificity: 0.6667


In [7]:
logits.grad = None

invalid_targets = torch.full_like(
    targets, -1
)

loss = combined(logits, invalid_targets)

loss.backward()

assert loss.item() == 0.0
assert (logits.grad == 0).all()

print("All-invalid mask test passed!")

All-invalid mask test passed!
